# Regression Analysis: Oil Price Shocks and Sovereign CDS Spreads

This notebook estimates panel regressions and threshold regressions to test
whether oil-exporting sovereigns exhibit **differential sensitivity** of CDS spread
movements to oil price shocks, relative to a control group of non-oil-exporting
emerging markets.

**Core identification strategy.** We interact Brent crude log-returns
($r_t^{\text{Brent}}$) with an oil-exporter indicator ($D_i^{\text{oil}}$) and test
whether the interaction coefficient is negative and significant — i.e., whether oil
exporters' CDS spreads widen *more* when oil prices fall than those of the control
group.

---

## 0. Imports

In [20]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import datetime
from linearmodels.panel import PanelOLS

## 1. Build the dataset

We load seven data sources and merge them on the weekly date index:

| Source | Contents |
|--------|----------|
| `Weekly_CDS.csv` | Sovereign 5-year CDS spreads (wide format, one column per country) |
| `oil_prices_datastream.csv` | Brent crude prices |
| `macro_risk_variables.csv` | DXY, US Treasury yields (2Y, 5Y, 10Y) |
| `VIXCLS.csv` | CBOE VIX index |
| `OVXCLS.csv` | CBOE Oil Volatility Index (OVX) |
| `Daily_FX_Rates.csv` | USD/local-currency exchange rates (wide, one column per country) |
| `mscicountryindex.csv` | MSCI country equity indices (wide, one column per country) |

CDS, oil, macro, VIX, and OVX are **global** variables (same value for every country
in a given week). FX rates and MSCI indices are **country-specific** and are joined
during the panel reshape step.

The sample starts on **2014-01-01** onward (post-OVX availability and coinciding with
the 2014 oil price collapse, which provides a natural starting point for studying
oil-sovereign transmission).

In [21]:
# ── Load individual datasets ────────────────────────────────────────────
CDS_data = pd.read_csv('../data/processed/CDS/Weekly_CDS.csv', parse_dates=['Date'], index_col='Date')
Oil_data = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv', parse_dates=['Date'], index_col='Date')
Macro    = pd.read_csv('../data/processed/Macroeconomic_variables/macro_risk_variables.csv', parse_dates=['Date'], index_col='Date')
VIX      = pd.read_csv('../data/processed/Macroeconomic_variables/VIXCLS.csv', parse_dates=['Date'], index_col='Date')
OVX      = pd.read_csv('../data/processed/Macroeconomic_variables/OVXCLS.csv', parse_dates=['date'], index_col='date')
# Country-specific data
FX_data   = pd.read_csv('../data/processed/Macroeconomic_variables/Daily_FX_Rates.csv', parse_dates=['Date'], index_col='Date')
MSCI_data = pd.read_csv('../data/processed/MSCI_indices/mscicountryindex.csv', parse_dates=['Date'], index_col='Date')

# ── Merge global variables on date index ────────────────────────────────
merged = (
    CDS_data
    .join(Oil_data, how='inner')
    .join(Macro,    how='inner')
    .join(VIX,      how='inner')
    .join(OVX,      how='inner')
)

# Filter to 2014+
merged = merged.loc[merged.index >= '2014-01-01'].copy()

print(f"Merged shape  : {merged.shape}")
print(f"Date range    : {merged.index.min().date()} -> {merged.index.max().date()}")
print(f"FX countries  : {FX_data.shape[1]}")
print(f"MSCI countries: {MSCI_data.shape[1]}")

/var/folders/wy/gjw_3_n51t748hfngpz4zf0w0000gn/T/ipykernel_29652/1796653235.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  Oil_data = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv', parse_dates=['Date'], index_col='Date')
/var/folders/wy/gjw_3_n51t748hfngpz4zf0w0000gn/T/ipykernel_29652/1796653235.py:4: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  Macro    = pd.read_csv('../data/processed/Macroeconomic_variables/macro_risk_variables.csv', parse_dates=['Date'], index_col='Date')


Merged shape  : (574, 83)
Date range    : 2014-01-03 -> 2024-12-27
FX countries  : 86
MSCI countries: 84


/var/folders/wy/gjw_3_n51t748hfngpz4zf0w0000gn/T/ipykernel_29652/1796653235.py:9: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  MSCI_data = pd.read_csv('../data/processed/MSCI_indices/mscicountryindex.csv', parse_dates=['Date'], index_col='Date')


## 2. Variable construction

All *price-level* variables are transformed to **log-returns**:

$$
r_t = \ln\!\left(\frac{P_t}{P_{t-1}}\right)
$$

This applies to: Brent, OVX, VIX, DXY, CDS spreads, FX rates, and MSCI indices.

The exception is US Treasury yields, which are already in percentage-point units;
for these we take the **first difference** ($\Delta y_t = y_t - y_{t-1}$), which is
standard in the fixed-income literature.

### Country groups

| Group | Countries |
|-------|-----------|
| **Oil exporters** | Saudi Arabia, Abu Dhabi, Dubai, Qatar, Colombia, Mexico, Brazil, Egypt, Malaysia |
| **Controls** | Indonesia, Philippines, Turkey, Chile, China, South Africa, South Korea, Thailand |

We reshape from wide to a balanced **panel** (long format) with one row per
country-week. Country-specific variables (CDS, FX, MSCI) are matched by country
name during the reshape. We then compute log-returns within each country group
and create interaction terms.

In [22]:
# ── Country groups ──────────────────────────────────────────────────────
oil_exporters = [
    'Saudi Arabia', 'Abu Dhabi', 'Dubai', 'Qatar', 'Colombia',
    'Mexico', 'Brazil', 'Egypt', 'Malaysia'
]
controls = [
    'Indonesia', 'Philippines', 'Turkey', 'Chile', 'China',
    'South Africa', 'South Korea', 'Thailand'
]

# ── Compute log-returns for global price-level variables ───────────────
merged['Oil_ret'] = np.log(merged['Brent']   / merged['Brent'].shift(1))
merged['OVX_chg'] = np.log(merged['OVXCLS']  / merged['OVXCLS'].shift(1))
merged['VIX_chg'] = np.log(merged['VIXCLS']  / merged['VIXCLS'].shift(1))
merged['DXY_chg'] = np.log(merged['DXY']     / merged['DXY'].shift(1))

# First differences for Treasury yields (basis-point changes)
merged['UST2Y_chg']  = merged['UST2Y'].diff()
merged['UST5Y_chg']  = merged['UST5Y'].diff()
merged['UST10Y_chg'] = merged['UST10Y'].diff()

# ── Reshape to long (panel) format ─────────────────────────────────────
global_vars = [
    'Brent', 'Oil_ret', 'OVXCLS', 'OVX_chg',
    'VIXCLS', 'VIX_chg', 'DXY_chg',
    'UST2Y_chg', 'UST5Y_chg', 'UST10Y_chg'
]

merged = merged.reset_index()          # bring Date back as column

# Align FX and MSCI to the same date index as merged
fx_aligned   = FX_data.reindex(pd.to_datetime(merged['Date'])).reset_index()
msci_aligned = MSCI_data.reindex(pd.to_datetime(merged['Date'])).reset_index()

panel_rows = []
for country in oil_exporters + controls:
    if country not in merged.columns:
        print(f"Warning: {country} not in CDS data -- skipping")
        continue

    temp = merged[['Date'] + global_vars + [country]].copy()
    temp = temp.rename(columns={country: 'CDS'})
    temp['Country']     = country
    temp['OilExporter'] = int(country in oil_exporters)

    # ── Match country-specific FX rate ─────────────────────────────────
    if country in fx_aligned.columns:
        temp['FX'] = fx_aligned[country].values
    else:
        print(f"  FX missing for {country} -- filling NaN")
        temp['FX'] = np.nan

    # ── Match country-specific MSCI index ──────────────────────────────
    if country in msci_aligned.columns:
        temp['MSCI'] = msci_aligned[country].values
    else:
        print(f"  MSCI missing for {country} -- filling NaN")
        temp['MSCI'] = np.nan

    panel_rows.append(temp)

panel = (
    pd.concat(panel_rows, ignore_index=True)
    .sort_values(['Country', 'Date'])
    .reset_index(drop=True)
)

# ── Country-specific log-returns (groupby to avoid cross-country diffs) ─
panel['CDS_ret'] = panel.groupby('Country')['CDS'].transform(
    lambda s: np.log(s / s.shift(1))
)
panel['FX_ret'] = panel.groupby('Country')['FX'].transform(
    lambda s: np.log(s / s.shift(1))
)
panel['MSCI_ret'] = panel.groupby('Country')['MSCI'].transform(
    lambda s: np.log(s / s.shift(1))
)

# ── Interaction terms ─────────────────────────────────────────────────
panel['OVX_x_Exporter']    = panel['OVX_chg'] * panel['OilExporter']
panel['Brent_x_Exporter']  = panel['Oil_ret'] * panel['OilExporter']

# ── Drop incomplete rows ──────────────────────────────────────────────
reg_vars = ['CDS_ret', 'OVX_chg', 'VIX_chg', 'DXY_chg',
            'UST2Y_chg', 'UST5Y_chg', 'UST10Y_chg']
panel = panel.dropna(subset=reg_vars).reset_index(drop=True)

# ── Report coverage ───────────────────────────────────────────────────
print(f"Panel shape  : {panel.shape}")
print(f"Countries    : {panel['Country'].nunique()}")
print(f"Weeks/country: ~{len(panel) // panel['Country'].nunique()}")

# Check FX/MSCI coverage
fx_coverage   = panel.groupby('Country')['FX_ret'].apply(lambda s: s.notna().mean())
msci_coverage = panel.groupby('Country')['MSCI_ret'].apply(lambda s: s.notna().mean())
coverage = pd.DataFrame({'FX_coverage': fx_coverage, 'MSCI_coverage': msci_coverage})
print("\nCountry-specific variable coverage (fraction non-NaN):")
print(coverage.to_string())
panel.head()

Panel shape  : (9129, 21)
Countries    : 17
Weeks/country: ~537

Country-specific variable coverage (fraction non-NaN):
              FX_coverage  MSCI_coverage
Country                                 
Abu Dhabi             1.0            1.0
Brazil                1.0            1.0
Chile                 1.0            1.0
China                 1.0            1.0
Colombia              1.0            1.0
Dubai                 1.0            1.0
Egypt                 1.0            1.0
Indonesia             1.0            1.0
Malaysia              1.0            1.0
Mexico                1.0            1.0
Philippines           1.0            1.0
Qatar                 1.0            1.0
Saudi Arabia          1.0            1.0
South Africa          1.0            1.0
South Korea           1.0            1.0
Thailand              1.0            1.0
Turkey                1.0            1.0


,Date,Brent,Oil_ret,OVXCLS,OVX_chg,VIXCLS,VIX_chg,DXY_chg,UST2Y_chg,UST5Y_chg,...,CDS,Country,OilExporter,FX,MSCI,CDS_ret,FX_ret,MSCI_ret,OVX_x_Exporter,Brent_x_Exporter
0,2014-01-10,106.33,-0.006935,19.47,-0.056416,12.14,-0.125260,-0.001610,-0.018,-0.103,...,55.85999,Abu Dhabi,1,3.6720,769.313,-0.008201,0.000000,0.002561,-0.056416,-0.006935
1,2014-01-17,106.89,0.005253,17.12,-0.128627,12.44,0.024411,0.007042,0.001,0.005,...,55.32999,Abu Dhabi,1,3.6720,801.257,-0.009533,0.000000,0.040684,-0.128627,0.005253
2,2014-01-24,107.45,0.005225,19.46,0.128114,18.14,0.377202,-0.009524,-0.031,-0.064,...,55.34999,Abu Dhabi,1,3.6729,828.858,0.000361,0.000245,0.033867,0.128114,0.005225
3,2014-01-31,107.13,-0.002983,20.35,0.044720,18.41,0.014775,0.010509,-0.008,-0.056,...,56.32999,Abu Dhabi,1,3.6728,822.438,0.017551,-0.000027,-0.007776,0.044720,-0.002983
4,2014-02-07,108.21,0.010031,19.03,-0.067064,15.29,-0.185695,-0.007654,-0.033,-0.048,...,56.32999,Abu Dhabi,1,3.6720,839.812,0.000000,-0.000218,0.020905,-0.067064,0.010031


## 3. Panel regressions

We estimate four nested specifications of the following general model:

$$
\Delta \ln \text{CDS}_{i,t}
= \alpha_i + \gamma_t
+ \beta_1 \, r_t^{\text{Brent}}
+ \beta_2 \,\bigl(r_t^{\text{Brent}} \times D_i^{\text{oil}}\bigr)
+ \boldsymbol{\delta}'\,\mathbf{X}_{i,t}
+ \varepsilon_{i,t}
$$

where:
- $\Delta \ln \text{CDS}_{i,t}$ is the weekly log-return of the 5-year CDS spread for
  country $i$,
- $r_t^{\text{Brent}} = \Delta \ln \text{Brent}_t$ is the weekly Brent crude log-return,
- $D_i^{\text{oil}}$ is a time-invariant dummy equal to 1 for oil exporters,
- $\mathbf{X}_{i,t}$ includes global controls
  ($\Delta \ln \text{VIX}_t$, $\Delta \ln \text{DXY}_t$, $\Delta \text{UST10Y}_t$)
  and country-specific controls ($r_{i,t}^{\text{MSCI}}$, $r_{i,t}^{\text{FX}}$),
- $\alpha_i$ are country fixed effects, $\gamma_t$ are time fixed effects.

The **coefficient of interest** is $\beta_2$: a negative and significant estimate
implies that oil exporters' CDS spreads widen *disproportionately* when oil prices
fall (since $r^{\text{Brent}} < 0 \Rightarrow \Delta\ln\text{CDS} > 0$ for exporters
more than for controls).

| Model | Fixed effects | Controls | Notes |
|-------|--------------|----------|-------|
| 1 | None | No | Baseline: Brent return + interaction only |
| 2 | None | Yes | Adds VIX, DXY, UST10Y, MSCI, FX |
| 3 | Country | Yes | Absorbs time-invariant country heterogeneity |
| 4 | Country + Time | — | Time FE absorb all global regressors; only the interaction survives |

All models use **standard errors clustered by country**.

### 3.0 Prepare panel index

In [23]:
# linearmodels requires a MultiIndex (entity, time)
reg_panel = panel.copy()
reg_panel['Date'] = pd.to_datetime(reg_panel['Date'])
reg_panel = reg_panel.set_index(['Country', 'Date'])

y = reg_panel['CDS_ret']

### 3.1 Model 1 — Baseline (no fixed effects)

$$
\Delta \ln \text{CDS}_{i,t}
= \beta_1\,r_t^{\text{Brent}}
+ \beta_2\,(r_t^{\text{Brent}} \times D_i^{\text{oil}})
+ \varepsilon_{i,t}
$$

In [24]:
X_1 = reg_panel[['Oil_ret', 'Brent_x_Exporter']]

model_1 = PanelOLS(y, X_1, entity_effects=False, time_effects=False)
results_1 = model_1.fit(cov_type='clustered', cluster_entity=True)
print(results_1.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                CDS_ret   R-squared:                        0.0745
Estimator:                   PanelOLS   R-squared (Between):             -0.0745
No. Observations:                9129   R-squared (Within):               0.0745
Date:                Sun, Feb 08 2026   R-squared (Overall):              0.0745
Time:                        14:40:23   Log-likelihood                  1.13e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      367.11
Entities:                          17   P-value                           0.0000
Avg Obs:                       537.00   Distribution:                  F(2,9127)
Min Obs:                       537.00                                           
Max Obs:                       537.00   F-statistic (robust):             85.824
                            

### 3.2 Model 2 — Baseline + global and country-specific controls (no fixed effects)

$$
\Delta \ln \text{CDS}_{i,t}
= \beta_1\,r_t^{\text{Brent}}
+ \beta_2\,(r_t^{\text{Brent}} \times D_i^{\text{oil}})
+ \beta_3\,\Delta \ln\text{VIX}_t
+ \beta_4\,\Delta \ln\text{DXY}_t
+ \beta_5\, r_{i,t}^{\text{MSCI}}
+ \beta_6\, r_{i,t}^{\text{FX}}
+ \beta_7\,\Delta\text{UST10Y}_t
+ \varepsilon_{i,t}
$$

In [25]:
X_2 = reg_panel[['Oil_ret', 'Brent_x_Exporter', 'VIX_chg',
                 'DXY_chg', 'MSCI_ret','FX_ret', 'UST10Y_chg']]

model_2 = PanelOLS(y, X_2, entity_effects=False, time_effects=False)
results_2 = model_2.fit(cov_type='clustered', cluster_entity=True)
print(results_2.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                CDS_ret   R-squared:                        0.3470
Estimator:                   PanelOLS   R-squared (Between):             -2.6831
No. Observations:                9129   R-squared (Within):               0.3476
Date:                Sun, Feb 08 2026   R-squared (Overall):              0.3470
Time:                        14:40:24   Log-likelihood                 1.289e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      692.55
Entities:                          17   P-value                           0.0000
Avg Obs:                       537.00   Distribution:                  F(7,9122)
Min Obs:                       537.00                                           
Max Obs:                       537.00   F-statistic (robust):             142.80
                            

### 3.3 Model 3 — Country fixed effects + controls

$$
\Delta \ln \text{CDS}_{i,t}
= \alpha_i
+ \beta_1\,r_t^{\text{Brent}}
+ \beta_2\,(r_t^{\text{Brent}} \times D_i^{\text{oil}})
+ \boldsymbol{\delta}'\mathbf{X}_{i,t}
+ \varepsilon_{i,t}
$$

Country fixed effects $\alpha_i$ absorb any time-invariant heterogeneity across
sovereigns (e.g., average credit quality, economic structure).

In [26]:
X_3 = reg_panel[['Oil_ret', 'Brent_x_Exporter', 'VIX_chg',
                 'DXY_chg', 'MSCI_ret','FX_ret', 'UST10Y_chg']]

model_3 = PanelOLS(y, X_3, entity_effects=True, time_effects=False)
results_3 = model_3.fit(cov_type='clustered', cluster_entity=True)
print(results_3.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                CDS_ret   R-squared:                        0.3476
Estimator:                   PanelOLS   R-squared (Between):             -2.7024
No. Observations:                9129   R-squared (Within):               0.3476
Date:                Sun, Feb 08 2026   R-squared (Overall):              0.3470
Time:                        14:40:25   Log-likelihood                  1.29e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      693.12
Entities:                          17   P-value                           0.0000
Avg Obs:                       537.00   Distribution:                  F(7,9105)
Min Obs:                       537.00                                           
Max Obs:                       537.00   F-statistic (robust):             140.84
                            

### 3.4 Model 4 — Country + time fixed effects

$$
\Delta \ln \text{CDS}_{i,t}
= \alpha_i + \gamma_t
+ \beta_2\,(r_t^{\text{Brent}} \times D_i^{\text{oil}})
+ \varepsilon_{i,t}
$$

Time fixed effects $\gamma_t$ absorb **all** week-specific global shocks (oil price
level, VIX, DXY, Treasuries). The only regressor with cross-sectional variation that
survives is the **interaction term** — making this the cleanest test of differential
sensitivity.

In [27]:
X_4 = reg_panel[['Brent_x_Exporter']]

model_4 = PanelOLS(y, X_4, entity_effects=True, time_effects=True)
results_4 = model_4.fit(cov_type='clustered', cluster_entity=True)
print(results_4.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                CDS_ret   R-squared:                        0.0012
Estimator:                   PanelOLS   R-squared (Between):             -0.0052
No. Observations:                9129   R-squared (Within):               0.0123
Date:                Sun, Feb 08 2026   R-squared (Overall):              0.0123
Time:                        14:40:26   Log-likelihood                 1.515e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      9.9410
Entities:                          17   P-value                           0.0016
Avg Obs:                       537.00   Distribution:                  F(1,8575)
Min Obs:                       537.00                                           
Max Obs:                       537.00   F-statistic (robust):             0.9574
                            

### 3.5 Comparison across specifications

We summarise the interaction coefficient $\hat{\beta}_2$ (`Brent_x_Exporter`) across
the four models to assess stability.

In [28]:
print(f"\n{'Model':<45} {'β':>10} {'SE':>10} {'p-value':>10} {'R²':>10}")
print("-" * 85)

models = [
    ('Model 1: Baseline',              results_1),
    ('Model 2: + Global Controls',     results_2),
    ('Model 3: + Country FE',          results_3),
    ('Model 4: + Country FE + Time FE', results_4),
]

for name, res in models:
    beta = res.params['Brent_x_Exporter']
    se   = res.std_errors['Brent_x_Exporter']
    pval = res.pvalues['Brent_x_Exporter']
    r2   = res.rsquared_within if hasattr(res, 'rsquared_within') else res.rsquared

    sig = ''
    if pval < 0.1:  sig = '*'
    if pval < 0.05: sig = '**'
    if pval < 0.01: sig = '***'

    print(f"{name:<45} {beta:>10.4f} {se:>10.4f} {pval:>10.4f} {r2:>10.4f} {sig}")

print("\n* p<0.1, ** p<0.05, *** p<0.01")
print("Standard errors clustered by country.")


Model                                                  β         SE    p-value         R²
-------------------------------------------------------------------------------------
Model 1: Baseline                                -0.0597     0.0592     0.3128     0.0745 
Model 2: + Global Controls                       -0.0396     0.0433     0.3612     0.3476 
Model 3: + Country FE                            -0.0396     0.0433     0.3612     0.3476 
Model 4: + Country FE + Time FE                  -0.0597     0.0610     0.3279     0.0123 

* p<0.1, ** p<0.05, *** p<0.01
Standard errors clustered by country.


## 4. Threshold regressions

The full-sample panel regressions test the *average* relationship. But our thesis
hypothesis specifically concerns **extreme** oil-market episodes — periods when
oil-calibrated jumps would be activated in a structural model.

To capture this nonlinearity we condition on the **OVX level** rather than OVX returns:
a high OVX level signals elevated oil-market uncertainty regardless of the weekly
change. We restrict the sample to weeks when the OVX level exceeds a given quantile
and re-estimate the panel model with Brent returns as the main regressor.

For each threshold $q \in \{80\%, 90\%, 95\%, 99\%\}$, we define:

$$
\mathcal{T}_q = \bigl\{ t : \text{OVX}_t > Q_q(\text{OVX}) \bigr\}
$$

and estimate on the restricted sample $\{(i,t) : t \in \mathcal{T}_q\}$:

$$
\Delta \ln \text{CDS}_{i,t}
= \alpha_i
+ \beta_1\,r_t^{\text{Brent}}
+ \beta_2\,(r_t^{\text{Brent}} \times D_i^{\text{oil}})
+ \boldsymbol{\delta}'\mathbf{X}_{i,t}
+ \varepsilon_{i,t}
$$

If the oil-exporter differential is driven by **tail events**, we should see
$|\hat{\beta}_2|$ increase in magnitude as the threshold becomes more extreme.

In [29]:
results_list = []

ovx_quantiles = {
    'Full Sample': None,
    'OVX > Q70':   panel['OVXCLS'].quantile(0.70),
    'OVX > Q80':   panel['OVXCLS'].quantile(0.80),
    'OVX > Q90':   panel['OVXCLS'].quantile(0.90),
    'OVX > Q95':   panel['OVXCLS'].quantile(0.95),
    'OVX > Q99':   panel['OVXCLS'].quantile(0.99),
}

for label, cutoff in ovx_quantiles.items():

    if cutoff is None:
        subset = panel.copy()
    else:
        subset = panel[panel['OVXCLS'] > cutoff].copy()

    subset['Date'] = pd.to_datetime(subset['Date'])
    subset = subset.set_index(['Country', 'Date'])

    y_sub = subset['CDS_ret']

    # ── Spec A: OVX as regressor ──────────────────────────────────────
    X = subset[['Oil_ret', 'Brent_x_Exporter', 'VIX_chg',
                 'DXY_chg', 'MSCI_ret','FX_ret', 'UST10Y_chg']]
    res = PanelOLS(y_sub, X,
                     entity_effects=True, time_effects=False,
                     drop_absorbed=True, check_rank=False
            ).fit(cov_type='clustered', cluster_entity=True)

    results_list.append({
        'Threshold': label,
        'Cutoff':    cutoff,
        'N':         len(y_sub),
        # Brent interaction
        'β_Brent':   res.params['Brent_x_Exporter'],
        'SE_Brent':  res.std_errors['Brent_x_Exporter'],
        'p_Brent':   res.pvalues['Brent_x_Exporter'],
    })

# ── Print comparison ──────────────────────────────────────────────────
print(f"{'Threshold':<15} {'N':>6} │ "
      f"{'β_Brent':>8} {'p':>7}")
print("-" * 65)

for row in results_list:
    sig_b = '***' if row['p_Brent'] < 0.01 else '**' if row['p_Brent'] < 0.05 else '*' if row['p_Brent'] < 0.1 else ''

    print(f"{row['Threshold']:<15} {row['N']:>6} │ "
          f"{row['β_Brent']:>8.4f} {row['p_Brent']:>6.3f}{sig_b:<3}")

print("\nBoth specs: Country FE + global controls, clustered SEs by country")
print("Threshold: weeks where OVX level > quantile")

Threshold            N │  β_Brent       p
-----------------------------------------------------------------
Full Sample       9129 │  -0.0396  0.361   
OVX > Q70         2720 │  -0.0641  0.214   
OVX > Q80         1819 │  -0.1109  0.037** 
OVX > Q90          901 │  -0.1727  0.002***
OVX > Q95          442 │  -0.2465  0.000***
OVX > Q99           85 │  -0.3570  0.002***

Both specs: Country FE + global controls, clustered SEs by country
Threshold: weeks where OVX level > quantile


## 5. Triple-interaction model

The threshold regressions in §4 re-estimate the model on progressively smaller
sub-samples. An alternative that uses the **full sample** while still testing for
regime-dependent sensitivity is a triple-interaction design.

We define a high-stress indicator:

$$
H_t = \mathbf{1}\!\bigl[\,\text{OVX}_t > Q_{90}(\text{OVX})\,\bigr]
$$

and estimate:

$$
\Delta \ln \text{CDS}_{i,t}
= \alpha_i
+ \beta_1\,r_t^{\text{Brent}}
+ \beta_2\,(r_t^{\text{Brent}} \times D_i^{\text{oil}})
+ \beta_3\,(r_t^{\text{Brent}} \times H_t)
+ \beta_4\,(r_t^{\text{Brent}} \times D_i^{\text{oil}} \times H_t)
+ \boldsymbol{\delta}'\mathbf{X}_{i,t}
+ \varepsilon_{i,t}
$$

**Interpretation of coefficients:**

| Coefficient | Measures |
|-------------|----------|
| $\beta_1$ | Brent→CDS sensitivity for controls in calm weeks |
| $\beta_1 + \beta_2$ | Brent→CDS sensitivity for exporters in calm weeks |
| $\beta_1 + \beta_3$ | Brent→CDS sensitivity for controls in high-OVX weeks |
| $\beta_1 + \beta_2 + \beta_3 + \beta_4$ | Brent→CDS sensitivity for exporters in high-OVX weeks |

The **coefficient of interest** is $\beta_4$: a negative and significant estimate
means oil exporters' CDS spreads are *extra* sensitive to Brent returns during
high-stress episodes, beyond the general amplification ($\beta_3$) that all
sovereigns experience.

This is the single-equation analogue of our §4 threshold regressions, and
directly motivates the oil-calibrated jump component in the structural model:
oil exporters have a regime-dependent channel that controls do not.

In [30]:
# ── 5. Triple-interaction: Brent × OilExporter × HighOVX ──────────────

# Define high-OVX regime indicator (top decile of OVX *level*)
ovx_q90 = panel['OVXCLS'].quantile(0.80)
panel['HighOVX'] = (panel['OVXCLS'] > ovx_q90).astype(int)

# Construct interaction terms
panel['Brent_x_HighOVX']            = panel['Oil_ret'] * panel['HighOVX']
panel['Brent_x_Exporter_x_HighOVX'] = panel['Oil_ret'] * panel['OilExporter'] * panel['HighOVX']

# ── Prepare panel index ──────────────────────────────────────────────
tri = panel.copy()
tri['Date'] = pd.to_datetime(tri['Date'])
tri = tri.set_index(['Country', 'Date'])

y_tri = tri['CDS_ret']

# ── Model 5a: Country FE + controls ─────────────────────────────────
X_5a = tri[['Oil_ret', 'Brent_x_Exporter',
            'Brent_x_HighOVX', 'Brent_x_Exporter_x_HighOVX',
            'VIX_chg', 'DXY_chg', 'MSCI_ret', 'FX_ret', 'UST10Y_chg']]

model_5a = PanelOLS(y_tri, X_5a,
                    entity_effects=True, time_effects=False,
                    drop_absorbed=True, check_rank=False)
results_5a = model_5a.fit(cov_type='clustered', cluster_entity=True)

# ── Model 5b: Country + Time FE (interaction terms only) ────────────
X_5b = tri[['Brent_x_Exporter',
            'Brent_x_Exporter_x_HighOVX']]

model_5b = PanelOLS(y_tri, X_5b,
                    entity_effects=True, time_effects=True,
                    drop_absorbed=True, check_rank=False)
results_5b = model_5b.fit(cov_type='clustered', cluster_entity=True)

# ── Print results side by side ───────────────────────────────────────
print("=" * 80)
print("TRIPLE-INTERACTION RESULTS")
print(f"OVX threshold (Q90): {ovx_q90:.2f}")
print(f"High-OVX weeks: {panel['HighOVX'].sum() // panel['Country'].nunique()}"
      f" / {panel.groupby('Country').size().iloc[0]}"
      f" ({100 * panel['HighOVX'].mean():.1f}%)")
print("=" * 80)

print(f"\n{'':45} {'Model 5a':>12} {'Model 5b':>12}")
print(f"{'':45} {'(Country FE)':>12} {'(Two-way FE)':>12}")
print("-" * 70)

coefs_to_show = [
    ('Oil_ret',                       'β₁  Brent'),
    ('Brent_x_Exporter',             'β₂  Brent × Exporter'),
    ('Brent_x_HighOVX',              'β₃  Brent × HighOVX'),
    ('Brent_x_Exporter_x_HighOVX',   'β₄  Brent × Exp × HighOVX'),
]

for var, label in coefs_to_show:
    parts = []
    for res in [results_5a, results_5b]:
        if var in res.params.index:
            b = res.params[var]
            p = res.pvalues[var]
            sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''
            parts.append(f"{b:>8.4f}{sig:<3}")
        else:
            parts.append(f"{'(absorbed)':>11}")
    print(f"{label:<45} {parts[0]:>12} {parts[1]:>12}")

# Controls
print("-" * 70)
controls_show = ['VIX_chg', 'DXY_chg', 'MSCI_ret', 'FX_ret', 'UST10Y_chg']
for var in controls_show:
    parts = []
    for res in [results_5a, results_5b]:
        if var in res.params.index:
            b = res.params[var]
            p = res.pvalues[var]
            sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''
            parts.append(f"{b:>8.4f}{sig:<3}")
        else:
            parts.append(f"{'(absorbed)':>11}")
    print(f"{var:<45} {parts[0]:>12} {parts[1]:>12}")

print("-" * 70)
r2_a = results_5a.rsquared_within if hasattr(results_5a, 'rsquared_within') else results_5a.rsquared
r2_b = results_5b.rsquared_within if hasattr(results_5b, 'rsquared_within') else results_5b.rsquared
print(f"{'R² (within)':<45} {r2_a:>11.4f}  {r2_b:>11.4f}")
print(f"{'N':<45} {results_5a.nobs:>11.0f}  {results_5b.nobs:>11.0f}")
print(f"{'Entity FE':<45} {'Yes':>11}  {'Yes':>11}")
print(f"{'Time FE':<45} {'No':>11}  {'Yes':>11}")

print("\n* p<0.1, ** p<0.05, *** p<0.01")
print("Standard errors clustered by country.")

# ── Marginal effects table ───────────────────────────────────────────
print("\n" + "=" * 80)
print("IMPLIED MARGINAL EFFECTS (Model 5a)")
print("=" * 80)

b1 = results_5a.params['Oil_ret']
b2 = results_5a.params['Brent_x_Exporter']
b3 = results_5a.params['Brent_x_HighOVX']
b4 = results_5a.params['Brent_x_Exporter_x_HighOVX']

rows = [
    ('Controls,  calm  (β₁)',           b1),
    ('Exporters, calm  (β₁+β₂)',        b1 + b2),
    ('Controls,  stress (β₁+β₃)',       b1 + b3),
    ('Exporters, stress (β₁+β₂+β₃+β₄)', b1 + b2 + b3 + b4),
]

print(f"\n{'Group / Regime':<45} {'dCDS/dBrent':>12}")
print("-" * 58)
for label, val in rows:
    print(f"{label:<45} {val:>12.4f}")

print(f"\n  Exporter premium (calm):   β₂         = {b2:>8.4f}")
print(f"  Exporter premium (stress): β₂ + β₄    = {b2 + b4:>8.4f}")
print(f"  Extra stress premium:      β₄         = {b4:>8.4f}")

TRIPLE-INTERACTION RESULTS
OVX threshold (Q90): 46.56
High-OVX weeks: 107 / 537 (19.9%)

                                                  Model 5a     Model 5b
                                              (Country FE) (Two-way FE)
----------------------------------------------------------------------
β₁  Brent                                       -0.0441      (absorbed)
β₂  Brent × Exporter                             0.0637*      0.0467   
β₃  Brent × HighOVX                             -0.0729**    (absorbed)
β₄  Brent × Exp × HighOVX                       -0.1754***   -0.1804***
----------------------------------------------------------------------
VIX_chg                                          0.1099***   (absorbed)
DXY_chg                                          1.2885***   (absorbed)
MSCI_ret                                        -0.7836***   (absorbed)
FX_ret                                          -0.0021      (absorbed)
UST10Y_chg                                      -

# Relative oil price and CDS spreads


The idea here would be to test if high oil prices means lower risk of defaults for oil exporters.

My idea for a regression. Countries have different credit spreads levels, calculate something like a reference spread level (like 2year rolling average), calculate CDS_now/CDS_reference, regress that on oil_price_now/oil_reference, add other controls to see if the relationship holds, test if it is stronger for exporters than for controls

In [31]:
# --- Parameters ---
WEEKS_2Y = 104          # ~2 years of weekly data
MIN_WEEKS = 52          # allow earlier estimation; set to 104 if you want strict 2y history

# --- 1) Reference levels (rolling means) ---
# Country-specific CDS reference (rolling mean within country)
panel["CDS_ref"] = panel.groupby("Country")["CDS"].transform(
    lambda s: s.rolling(WEEKS_2Y, min_periods=MIN_WEEKS).mean()
)

panel["FX_ref"] = panel.groupby("Country")["FX"].transform(
    lambda s: s.rolling(WEEKS_2Y, min_periods=MIN_WEEKS).mean()
)

panel["MSCI_ref"] = panel.groupby("Country")["MSCI"].transform(
    lambda s: s.rolling(WEEKS_2Y, min_periods=MIN_WEEKS).mean()
)

# Global Brent reference (rolling mean by date; Brent is common across countries)
brent_by_date = (
    panel[["Date", "Brent"]]
    .drop_duplicates("Date")
    .sort_values("Date")
    .reset_index(drop=True)
)
brent_by_date["Brent_ref"] = brent_by_date["Brent"].rolling(WEEKS_2Y, min_periods=MIN_WEEKS).mean()

panel = panel.merge(brent_by_date[["Date", "Brent_ref"]], on="Date", how="left")

# --- 2) Relative variables (log ratios are scale-free and symmetric) ---
panel["CDS_rel"]   = (panel["CDS"]   / panel["CDS_ref"]) * 100
panel["Brent_rel"] = (panel["Brent"] / panel["Brent_ref"]) * 100

# Exporter interaction (the key coefficient if you include time FE)
panel["Brent_rel_x_Exporter"] = panel["Brent_rel"] * panel["OilExporter"]

# Drop rows where ratios are undefined
df_rel = panel.dropna(subset=["CDS_rel", "Brent_rel", "Brent_rel_x_Exporter"]).copy()

# --- 3) Panel index for PanelOLS ---
rel = df_rel.copy()
rel["Date"] = pd.to_datetime(rel["Date"])
rel = rel.set_index(["Country", "Date"]).sort_index()

y_rel = rel["CDS_rel"]

# --- 4) Spec A: Country FE (keeps oil-relative level + global controls) ---
# NOTE: With only country FE, you can include ln_Brent_rel and global controls.
X_A_cols = [
    "Brent_rel",
    "Brent_rel_x_Exporter",
    # controls already constructed earlier in the notebook
    "VIXCLS", "OVXCLS",          # levels (global)
    "DXY", "UST10Y",     # changes (global)
    "MSCI_rel", "FX_rel"         # country-specific
]
X_A_cols = [c for c in X_A_cols if c in rel.columns]  # be robust if something is missing

X_A = rel[X_A_cols]
res_A = PanelOLS(y_rel, X_A, entity_effects=True, time_effects=False).fit(
    cov_type="clustered", cluster_entity=True
)

# --- 5) Spec B: Country + Time FE (cleanest exporter-vs-control differential test) ---
# With time FE, any regressor that is the same across countries in a week is absorbed.
# So we OMIT ln_Brent_rel and global controls; keep only the exporter interaction + country-specific controls.
X_B_cols = [
    "ln_Brent_rel_x_Exporter",
    "MSCI_ret", "FX_ret"
]
X_B_cols = [c for c in X_B_cols if c in rel.columns]

X_B = rel[X_B_cols]
res_B = PanelOLS(y_rel, X_B, entity_effects=True, time_effects=True).fit(
    cov_type="clustered", cluster_entity=True
)

# --- 6) Output ---
print("\n================ Relative-level regression: CDS/CDS_ref on Brent/Brent_ref ================\n")
print("Spec A: Country FE (keeps Brent_rel + global controls)\n")
print(res_A.summary)

print("\n\nSpec B: Country + Time FE (identifies only differential exporter effect)\n")
print(res_B.summary)

# Compact comparison for the key coefficient
key = "ln_Brent_rel_x_Exporter"
print("\n\nKey coefficient comparison (interaction):")
print(f"{'Model':<35} {'β':>10} {'SE':>10} {'p-value':>10}")
print("-" * 70)

for name, res in [("Spec A: Country FE", res_A), ("Spec B: Country+Time FE", res_B)]:
    if key in res.params.index:
        beta = res.params[key]
        se   = res.std_errors[key]
        pval = res.pvalues[key]
        print(f"{name:<35} {beta:>10.4f} {se:>10.4f} {pval:>10.4f}")
    else:
        print(f"{name:<35} {'(dropped)':>10} {'':>10} {'':>10}")

# (Optional) quick sanity: average CDS_rel by group
grp = df_rel.groupby("OilExporter")["CDS_rel"].mean()
print("\nSanity check: mean CDS_rel by group (0=control, 1=exporter):")
print(grp)



================ Relative-level regression: log(CDS/CDS_ref) on log(Brent/Brent_ref) ================

Spec A: Country FE (keeps ln_Brent_rel + global controls)

                          PanelOLS Estimation Summary                           
Dep. Variable:                CDS_rel   R-squared:                        0.3360
Estimator:                   PanelOLS   R-squared (Between):              0.6033
No. Observations:                8262   R-squared (Within):               0.3360
Date:                Sun, Feb 08 2026   R-squared (Overall):              0.5781
Time:                        14:40:30   Log-likelihood                -3.871e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      1042.4
Entities:                          17   P-value                           0.0000
Avg Obs:                       486.00   Distribution:                  F(4,8241)
Min Obs:                   